In [ ]:
# Step 1.1
#Upload Pre-Trade Price File
#User selected the pre-trade file
#Name of the file:

from google.colab import files
case_pre = files.upload()

In [ ]:
# Step 1.2
# Upload Post-Trade Price File
# User select the post-trade file
# Name of the file:

from google.colab import files
case_post = files.upload()

In [ ]:
#Step 2: Import Packages
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import scipy.stats
from scipy.optimize import minimize

In [ ]:
#Step 3: Set I/O folders
MIDataFile_Post = '/content/Case Study 01 - Post Trade Analysis.csv'
MIDataFile_Pre = '/content/Case Study 02 - Pre Trade Analysis.csv'

In [ ]:
#Step 4: This is the parameter of the Market Impact model
a1=883.58722
a2=0.35408
a3=0.755684
a4=0.826155
b1=0.963532
MIParams=[a1, a2, a3, a4, b1]
print(MIParams)

In [ ]:
#Step 5.1:
#TCA Functions: Pre-Trade Functions

# Market Impact
def MI_calc(Size, Volatility, POV, MIParams):
    a1, a2, a3, a4, b1 = MIParams
    mi = (a1*Size**a2*Volatility**a3) * (b1*POV**a4 + (1-b1))
    return mi

# Timing Risk
def TR_calc(Size, Volatility, POV):
    tr = Volatility * ((1/3*1/250*Size*(1-POV)/POV )**0.5) * 10**4
    return tr

# Price Appreciation
def PA_calc(Size, AlphaBp, POV, Side):
    pa = Side * 1/2 * AlphaBp * Size * ((1 - POV)/POV)
    return pa

# POV to Trade Time
def POVToTime_calc(Size, POV):
    tt = Size * ((1-POV)/POV)
    return tt

# Trade Time To POV
def TimeToPOV_calc(Size, Time):
    pov = Size / (Time+Size)
    return pov


In [ ]:
#Step 5.2
#TCA Functions: Post-Trade Functions

def IS_Cost_calc(Pavg, Pd, P0, Pn, S, X, R, comm, Side):
    Delay = S * (P0 - Pd) * Side
    Exec = X * (Pavg-P0) * Side
    Opport = R * (Pn - P0) * Side
    Fixed = X * comm
    ImplShortFall = Delay + Exec + Opport + Fixed
    return int(round(ImplShortFall,0)), int(round(Delay, 0)), int(round(Exec, 0)), int(round(Opport, 0)), int(round(Fixed, 0))

#Paper Return
def IS_PaperReturn_calc(S,Pd,Pn, Side):
    IS_PaperReturn = Side * S * (Pn - Pd)
    return int(round(IS_PaperReturn, 0))

#Actual Net Portfolio Return
def IS_ActualReturn_calc (X, Pavg, Pn, fixed, Side):
    IS_ActualReturn = Side * X * (Pn - Pavg) - fixed * X
    return round(IS_ActualReturn, 0)

#Arrival Cost
def Arrival_Cost_calc (Pavg, P0, Side):
    arrival_Cost = Side * (Pavg - P0)/P0 * 10**4
    return arrival_Cost

# VWAP Slippage
def VWAP_Slippage_calc (Pavg, VWAP, Side):
    vwap_slippage = Side * (Pavg - VWAP)/VWAP * 10**4
    return vwap_slippage

#Benchmark Cost
def Benchmark_Cost_calc (Pavg, Benchmark, Side):
    benchmark_cost = Side * (Pavg - Benchmark)/Benchmark * 10**4
    return benchmark_cost

#Value-Add
#ExpCost = MI + PA
#Arrival Cost
#TR

def ValueAdd_calc(arrival_Cost, MI, TR):
    value_add = MI - arrival_Cost
    zscore = value_add / TR
    return value_add, zscore
[]
# RPM (Relative Performance Measure)
def RPM_calc (price, volume, Pavg, side):
    total_volume = 0
    volume_at_avg = 0
    volume_larger_avg = 0
    volume_smaller_avg = 0
    for i in range(0, len(price)):
        if price[i] > Pavg:
            volume_larger_avg += volume[i]
        elif price[i] < Pavg:
            volume_smaller_avg += volume[i]
        elif price[i] == Pavg:
             volume_at_avg += volume[i]
        total_volume += volume[i]
    if side == 1:
        RPM = (volume_larger_avg + (0.5 * volume_at_avg)) / total_volume
    else:
        RPM = (volume_smaller_avg + (0.5 * volume_at_avg)) / total_volume
    return RPM;

In [ ]:
# Step 5.3:
# Traders Dilemma Objective Function
def TradersDilemma_OptCalc(POV, Size, Volatility, Lambda, MIParams):
    MI = MI_calc(Size, Volatility, POV, MIParams)
    TR = TR_calc(Size, Volatility, POV)
    ObjFun = MI + (Lambda*TR)
    return ObjFun

# Minimize Cost Objective Function
def MinimizeCost_OptCalc(POV, Size, Volatility, AlphaBp, Side, MIParams):
    MI = MI_calc(Size, Volatility, POV, MIParams)
    PA = PA_calc(Size, AlphaBp, POV, Side)
    MinCost = MI + PA
    return MinCost

# Price Improvement Objective Function
def PriceImprov_OptCalc(POV, Size, Volatility, Bid, MIParams):
    MI = MI_calc(Size, Volatility, POV, MIParams)
    TR = TR_calc(Size, Volatility, POV)
    PriceImprov = (MI - Bid) / TR
    return PriceImprov

Case Study 1: Pre-Trade Analysis

In [ ]:
# Step 6.1: Load Market Impact Data
MIData_Pre=pd.read_csv(MIDataFile_Pre, sep=',')
print(MIData_Pre)

# Step 6.2: Define Variables

Number = MIData_Pre['Number']
Trade_Date = MIData_Pre['Trade Date']
Symbol = MIData_Pre['Symbol']
Shares = MIData_Pre['Shares']
Price = MIData_Pre['Price']
Volatility = MIData_Pre['Volatility']
ADV = MIData_Pre['ADV']
Beta = MIData_Pre['Beta']

SSide = MIData_Pre['Side']
  #SSide = 'Buy' or 'Sell'

#Convert to 1 for Buy -1 for Sell
Side = []
for i in range(len(Symbol)):
  if SSide[i] == 'Buy':
    Side.append(1)
  else:
    Side.append(-1)

print(Side)


In [ ]:
#Step 6.3: Using POV = 10%
MI_10 = []
TR_10 = []
PA_10 = []

for i in range (len(Symbol)):
  alphaBp = Beta[i]*50
  size = Shares[i] / ADV[i]
  volatility = Volatility[i]
  side=Side[i]
  pov = 0.1

  mi_10 = MI_calc(size, volatility, pov, MIParams)
  tr_10 = TR_calc(size, volatility, pov)
  pa_10 = PA_calc(size, alphaBp, pov, side)

  MI_10.append(mi_10)
  TR_10.append(tr_10)
  PA_10.append(pa_10)


In [ ]:
#Part 6.4: Using Trade Time = 0.75 days
MI_75 = []
TR_75 = []
PA_75 = []

for i in range (len(Symbol)):
  alphaBp = Beta[i]*50
  size = Shares[i] / ADV[i]
  volatility = Volatility[i]
  time = 0.75
  side=Side[i]
  pov = TimeToPOV_calc(size, time)

  mi_75 = MI_calc(size, volatility, pov, MIParams)
  tr_75 = TR_calc(size, volatility, pov)
  pa_75 = PA_calc(size, alphaBp, pov, side)

  MI_75.append(mi_75)
  TR_75.append(tr_75)
  PA_75.append(pa_75)

In [ ]:
#Part 6.5: Trader Dilenma
POV_TD = []

# Using the Traders Dilemma Objective Function

# ----- Optimization Code
for i in range (len(Symbol)):
  Lambda = 0.35
  size = Shares[i] / ADV[i]
  volatility = Volatility[i]

  # Upper and Lower Bounds
  bound1 = [(0.00001, 0.9999)]
  bounds = (bound1)

  #initial guess
  x0 = 0.5
  # solve the minimization problem
  results = minimize(TradersDilemma_OptCalc, x0, bounds = bounds, args = (size, volatility, Lambda, MIParams))
  pov = results.x[0]
  POV_TD.append(pov)

print(POV_TD)


In [ ]:
#Part 6.6: Mininmize Cost
# Min MI + PA * TR
# Order Characteristics

POV_MinCost = []
# Using the Minimize Cost Objective Function

for i in range (len(Symbol)):
  # ----- Optimization Code
  size = Shares[i] / ADV[i]
  volatility = Volatility[i]
  alphaBp = Beta[i]*50
  side=Side[i]

  # Upper and Lower Bounds
  bound1 = [(0.00001, 0.9999)]
  bounds = (bound1)

  #initial guess
  x0 = 0.5

  # solve the minimization problem
  results = minimize(MinimizeCost_OptCalc, x0, bounds = bounds, args = (size, volatility, alphaBp, side, MIParams))

  pov_MinCost=results.x[0]
  POV_MinCost.append(pov_MinCost)

print(POV_MinCost)

In [ ]:
# Part 6.7: Price Improvement

POV_PriceImprov = []

# Using Price Improvement Objective Function

for i in range (len(Symbol)):
  # ----- Optimization Code
  size = Shares[i] / ADV[i]
  volatility = Volatility[i]
  Bid = 100

  # Upper and Lower Bounds
  bound1 = [(0.00001, 0.9999)]
  bounds = (bound1)

  #initial guess
  x0 = 0.5
  # solve the minimization problem
  results = minimize(PriceImprov_OptCalc, x0, bounds = bounds, args = (size, volatility, Bid, MIParams))
  pov_PriceImprov = results.x[0]
  POV_PriceImprov.append(pov_PriceImprov)


print(POV_PriceImprov)

In [ ]:
#Step 6.8: Output
OutputFile_Pre = 'PreTrade_Results.csv'
DataResults_Pre = list(zip(MI_10, TR_10, PA_10, MI_75, TR_75, PA_75, POV_TD, POV_MinCost, POV_PriceImprov))
DataResults_df = pd.DataFrame(DataResults_Pre, columns=['MI(POV = 10%)', 'TR(POV = 10%)', 'PA(POV = 10%)', 'MI(Time = 0.75 Day)', 'TR(Time = 0.75 Day)', 'PA(Time = 0.75 Day)', 'POV-Trader Dilenma', 'POV-Min Cost', 'POV-Price Improvement'])
DataResults_df = DataResults_df.round(4)
DataResults_df.to_csv(OutputFile_Pre, index = False, header = True)

print(DataResults_df)

CASE STUDY 2: Post-Trade

In [ ]:
#Step 7.1: Load Market Impact Data
MIData_Post=pd.read_csv(MIDataFile_Post, sep=',')
print(MIData_Post)

# Step 7.2: Define Variables
Pd=MIData_Post['Pd']
P0=MIData_Post['P0']
Pavg=MIData_Post['P_avg']
Pn=MIData_Post['Pn']
VWAP = MIData_Post['VWAP']

Symbol = MIData_Post['Symbol']
Trade_Date = MIData_Post['Trade Date']
Broker = MIData_Post['Broker']
ADV = MIData_Post['ADV']
Volatility = MIData_Post['Volatility']
Beta = MIData_Post['Beta']
MktCap = MIData_Post['Mkt Cap (MM)']

SSide = MIData_Post['Side']
  #SSide = 'Buy' or 'Sell'

Order = MIData_Post['Order Shares']
Executed = MIData_Post['Executed Shares']
POV = MIData_Post['POV']

Residual = MIData_Post['Order Shares'] - MIData_Post['Executed Shares']
comm = 0.01
Size = Order/ADV

#Convert to 1 for Buy -1 for Sell
Side = []
for i in range(len(Symbol)):
  if SSide[i] == 'Buy':
    Side.append(1)
  else:
    Side.append(-1)

print(Side)

MI=[]
for i in range(len(Symbol)):
  MI.append(MI_calc(Size[i], Volatility[i], POV[i], MIParams))

TR = []
for i in range(len(Symbol)):
  TR.append(TR_calc(Size[i], Volatility[i], POV[i]))


#Define Arrays to save regression results for each iteration
IS, Delay, Exec, Oppor, Fixed, Arrival, VWAP_Slippage, ValueAdd, ZScore = ([] for i in range(9))

In [ ]:
#Step 7.3:
for i in range (len(Symbol)):

  #Implementation Shortfall:

  implShortFall, delay, exec, oppor, fixed = IS_Cost_calc(Pavg[i], Pd[i], P0[i], Pn[i], Order[i], Executed[i], Residual[i] , comm, Side[i])
  IS.append(implShortFall)
  Delay.append(delay)
  Exec.append(exec)
  Oppor.append(oppor)
  Fixed.append(fixed)

  #Arrival Cost
  arrival = Arrival_Cost_calc (Pavg[i], P0[i], Side[i])
  Arrival.append(arrival)


  #VWAP Slippage
  vwap_slippage=VWAP_Slippage_calc (Pavg[i], VWAP[i], Side[i])
  VWAP_Slippage.append(vwap_slippage)

  #Value-Add
  #need to calculate MI Cost and TR
  #2 output values, value-add, and z-score
  #requires MI and TR of the order

  value_add, zscore = ValueAdd_calc(Arrival[i], MI[i], TR[i])
  ValueAdd.append(value_add)
  ZScore.append(zscore)


#Step 7.4: Output
OutputFile_Post = 'PostTrade_Results.csv'
DataResults = list(zip(IS, Delay, Exec, Oppor, Fixed, Arrival, VWAP_Slippage, ValueAdd, ZScore))
DataResults_df_2 = pd.DataFrame(DataResults, columns=['IS', 'Delay', 'Exec', 'Oppor', 'Fixed', 'Arrival', 'VWAP_Slipapage', 'ValueAdd', 'ZScore'])
DataResults_df.to_csv(OutputFile_Post, index = False, header = True)

print(DataResults_df)